# Sistema multiagente de soporte interno — Andina Obras S.A.C.

Proyecto Integrador del Módulo 3 — AI Engineering (Henry).

Un **orquestador** clasifica la consulta de un colaborador y la enruta al **agente RAG especializado** del área correspondiente (Recursos Humanos, Mesa de Ayuda de TI o Finanzas), que responde usando únicamente la documentación interna de esa área. Todo el flujo está implementado con **LangChain** y trazado con **Langfuse**.


```
Consulta del usuario
        │
        ▼
  Orquestador ──► hr | tech | finance | out_of_scope
        │
        ├─► Agente HR      (RAG sobre data/hr_docs)
        ├─► Agente Tech    (RAG sobre data/tech_docs)
        └─► Agente Finance (RAG sobre data/finance_docs)
        │
        ▼
  Respuesta con fuentes  ──►  Trace en Langfuse
```

> Andina Obras es una empresa ficticia; sus políticas fueron creadas con fines de formación.

## 1. Setup e imports

Se carga la configuración central (`src/config.py`), que lee las claves del archivo `.env`, y se verifica que todo esté listo antes de ejecutar el sistema.

In [1]:
from src import config
from src.tracing import get_langfuse

# Falla con un mensaje claro si falta alguna clave en el .env (fail-fast)
config.validate_config()

print("Modelo LLM       :", config.LLM_MODEL)
print("Modelo embeddings:", config.EMBEDDING_MODEL)
print("Dominios         :", list(config.DOMAINS))
print("TOP_K            :", config.TOP_K)
print("Langfuse conectado:", get_langfuse().auth_check())

Modelo LLM       : gpt-5.4-mini
Modelo embeddings: text-embedding-3-small
Dominios         : ['hr', 'tech', 'finance']
TOP_K            : 3
Langfuse conectado: True


## 2. Carga de documentos y vector stores

Cada agente tiene su propia base de conocimiento en `data/<dominio>_docs/` (archivos `.md`). El pipeline de ingesta (`src/ingest.py`) divide cada documento por secciones (`##`), genera los embeddings con OpenAI y los guarda en una colección de **Chroma** por dominio.

La ingesta se ejecuta una sola vez desde la terminal, antes de usar el sistema:

    python -m src.ingest

Aquí se verifica que las tres colecciones existen y superan el mínimo de 50 chunks.

In [2]:
from src.vectorstore import get_vectorstore

for domain in config.DOMAINS:
    store = get_vectorstore(domain)
    total = len(store.get()["ids"])
    estado = "OK" if total >= 50 else "MENOS DE 50"
    print(f"{domain:<8} {total:>3} chunks  [{estado}]")

hr        54 chunks  [OK]
tech      54 chunks  [OK]
finance   55 chunks  [OK]


In [3]:
# Ejemplo: un chunk con su metadata de procedencia
ejemplo = get_vectorstore("finance").get(limit=1)

print("Metadata:")
for clave, valor in ejemplo["metadatas"][0].items():
    print(f"  {clave}: {valor}")

print("\nTexto del chunk:")
print(ejemplo["documents"][0])

Metadata:
  chunk_id: finance/caja_chica_de_obra/000
  section: Qué es la caja chica de obra
  source: caja_chica_de_obra.md
  domain: finance
  title: Caja chica de obra - Andina Obras S.A.C.

Texto del chunk:
Caja chica de obra - Andina Obras S.A.C.
## Qué es la caja chica de obra
La caja chica es un fondo de dinero en efectivo que se asigna a cada obra para cubrir gastos menores, urgentes y no previsibles que no pueden esperar al proceso normal de compras. Por ejemplo, la compra de clavos o alambre cuando se agotan en almacén, el pago de una movilidad urgente o la compra de agua para el personal. La caja chica no reemplaza al proceso de compras: las compras planificadas o de mayor valor siempre deben hacerse mediante orden de compra.


## 3. Definición de agentes

Los tres agentes especialistas comparten una misma clase, `RAGAgent` (`src/agents/rag_agent.py`), que implementa el flujo RAG: **recuperar** los 3 chunks más parecidos de su colección, **armar el contexto** con su procedencia y **generar** la respuesta con el LLM, citando la fuente.

Cada agente (`hr_agent.py`, `tech_agent.py`, `finance_agent.py`) es una instancia de esa clase con su propio dominio y rol, para no repetir lógica.

A continuación se prueba cada agente directamente, sin pasar por el orquestador.

In [4]:
from src.agents.finance_agent import finance_agent
from src.agents.hr_agent import hr_agent
from src.agents.tech_agent import tech_agent

pruebas = [
    (hr_agent, "¿Cuántos días de licencia por paternidad me corresponden?"),
    (tech_agent, "Olvidé mi contraseña del correo, ¿cómo la recupero?"),
    (finance_agent, "¿Cuánto es el viático diario si viajo a la obra de Arequipa?"),
]

for agente, pregunta in pruebas:
    respuesta = agente.answer(pregunta)
    print(f"[{agente.domain.upper()}] {pregunta}")
    print(respuesta.answer)
    print("Fuentes:", [f"{s.section} ({s.score})" for s in respuesta.sources])
    print("-" * 80)

[HR] ¿Cuántos días de licencia por paternidad me corresponden?
Te corresponden **10 días calendario consecutivos** de licencia por paternidad, contados desde el nacimiento de tu hijo o hija. La licencia se registra en **Andina360** como “Licencia por paternidad” y debe sustentarse con el **acta de nacimiento** dentro de los **15 días posteriores** al inicio.  
**Fuente: Licencia por paternidad**
Fuentes: ['Licencia por paternidad (0.673)', 'Licencia por maternidad (0.618)', 'Licencia por fallecimiento de un familiar (0.503)']
--------------------------------------------------------------------------------
[TECH] Olvidé mi contraseña del correo, ¿cómo la recupero?
Puedes restablecerla tú mismo en **clave.andinaobras.pe** usando el código que recibes en el celular registrado en tu cuenta; este autoservicio está disponible las 24 horas. Si no tienes acceso a tu celular registrado o la opción no funciona, debes llamar al **anexo 2020** para que verifiquen tu identidad y generen una contras

In [5]:
# Fallo seguro: una pregunta que NO es de RR. HH. enviada al agente de RR. HH.
respuesta = hr_agent.answer("¿Cómo configuro la VPN en mi laptop?")
print(respuesta.answer)

No encontré esa información en la documentación disponible. La configuración de la VPN y el acceso remoto a los sistemas se gestionan con la Mesa de Ayuda de TI, no con Recursos Humanos; te sugiero consultar a esa área.  
Fuente: Requisitos para trabajar de forma remota.


## 4. Orquestador y enrutamiento condicional

El orquestador (`src/agents/orchestrator.py`) clasifica cada consulta en una de cuatro categorías: `hr`, `tech`, `finance` u `out_of_scope`. Usa un LLM con **salida estructurada** (`with_structured_output` + Pydantic), lo que garantiza que siempre devuelva una categoría válida junto con su justificación.

Las reglas del prompt (`prompts/orchestrator_system.txt`) clasifican según el **área dueña del tema** y no según palabras sueltas. Por ejemplo, si una aplicación no funciona, es un problema de TI aunque la aplicación pertenezca a otra área.

El enrutamiento condicional se hace en `src/multi_agent_system.py` con un diccionario `intent -> agente`. Las consultas `out_of_scope` reciben un mensaje fijo, sin llamar a ningún agente.

In [6]:
from src.agents.orchestrator import classify

consultas = [
    "¿Cuántos días de vacaciones me tocan?",
    "La app de Andina360 no me deja marcar mi asistencia",
    "En Andina360 mi saldo de vacaciones aparece con menos días de los que tengo",
    "Me depositaron menos de lo normal este mes, ¿a quién reclamo?",
    "¿Cómo calculo el volumen de concreto para una losa?",
]

for consulta in consultas:
    decision = classify(consulta)
    print(f"{consulta}")
    print(f"  -> {decision.intent}  (confianza {decision.confidence:.2f})")
    print(f"  Razonamiento: {decision.reasoning}")
    print()

¿Cuántos días de vacaciones me tocan?
  -> hr  (confianza 0.99)
  Razonamiento: Consulta sobre vacaciones, que corresponde a Recursos Humanos.

La app de Andina360 no me deja marcar mi asistencia
  -> tech  (confianza 0.99)
  Razonamiento: Es un problema de funcionamiento de una aplicación de asistencia (Andina360) que impide trabajar, por lo que corresponde a Mesa de Ayuda de TI.

En Andina360 mi saldo de vacaciones aparece con menos días de los que tengo
  -> hr  (confianza 0.96)
  Razonamiento: Es una consulta sobre saldo de vacaciones, que corresponde a Recursos Humanos; aunque se vea en Andina360, el dato es de HR y la aplicación funciona.

Me depositaron menos de lo normal este mes, ¿a quién reclamo?
  -> hr  (confianza 0.96)
  Razonamiento: La consulta trata sobre un depósito de sueldo menor al habitual, tema de remuneraciones y boleta de pago, que corresponde a Recursos Humanos.

¿Cómo calculo el volumen de concreto para una losa?
  -> out_of_scope  (confianza 0.99)
  Razonamie

## 5. Pruebas y ejemplos

Se ejecutan las 17 consultas de `test_queries.json` por el **sistema completo** (orquestador + agente). Las consultas cubren los tres agentes y casos difíciles en cinco categorías:

| Categoría | Qué prueba |
|---|---|
| directa | Preguntas típicas de cada área |
| informal | Lenguaje coloquial ("¿cuándo me cae la grati?") |
| ambigua | Consultas que parecen de un área pero pertenecen a otra |
| fuera_de_alcance | Consultas que ningún agente debe responder |
| multi_intencion | Dos temas en una misma consulta (limitación conocida) |

Cada consulta genera su traza en Langfuse con el tag `pruebas-v2`, para poder filtrarlas juntas.

Además, cada respuesta es revisada por un agente evaluador que le asigna una calidad del 1 al 10.

In [2]:
import json
from src import config
from src.multi_agent_system import handle_query
from src.tracing import get_langfuse

casos = json.loads(config.TEST_QUERIES_PATH.read_text(encoding="utf-8"))

resultados = []
for caso in casos:
    respuesta = handle_query(
        caso["query"],
        tags=["pruebas-v2", f"esperado-{caso['expected_intent']}"],
    )
    acierto = respuesta.intent == caso["expected_intent"]
    resultados.append({**caso, "obtenido": respuesta.intent, "acierto": acierto,
                       "respuesta": respuesta.answer, "evaluacion": respuesta.evaluation})

    marca = "✅" if acierto else "❌"
    print(f"{marca} #{caso['id']:>2} [{caso['category']:<16}] esperado={caso['expected_intent']:<12} "
          f"obtenido={respuesta.intent:<12} | {caso['query']}")

get_langfuse().flush()

✅ # 1 [directa         ] esperado=hr           obtenido=hr           | ¿Con cuánta anticipación debo pedir mis vacaciones?
✅ # 2 [directa         ] esperado=hr           obtenido=hr           | ¿Cuántos días de licencia por paternidad me corresponden?
✅ # 3 [directa         ] esperado=tech         obtenido=tech         | Olvidé mi contraseña del correo, ¿cómo la recupero?
✅ # 4 [directa         ] esperado=tech         obtenido=tech         | La VPN se queda en 'Conectando...' y no avanza
✅ # 5 [directa         ] esperado=finance      obtenido=finance      | ¿Cuál es el tope de movilidad sin comprobante?
✅ # 6 [directa         ] esperado=finance      obtenido=finance      | ¿Cuánto es el viático diario si viajo a la obra de Arequipa?
✅ # 7 [informal        ] esperado=hr           obtenido=hr           | oye cuando me cae la grati de julio??
✅ # 8 [informal        ] esperado=tech         obtenido=tech         | no me agarra la huella en el reloj de la obra
✅ # 9 [ambigua         ] espera

In [8]:
from collections import defaultdict

total = len(resultados)
aciertos = sum(r["acierto"] for r in resultados)
print(f"Routing accuracy: {aciertos}/{total} = {aciertos / total:.0%}\n")

por_categoria = defaultdict(lambda: [0, 0])
for r in resultados:
    por_categoria[r["category"]][0] += r["acierto"]
    por_categoria[r["category"]][1] += 1

for categoria, (ok, n) in por_categoria.items():
    print(f"  {categoria:<18} {ok}/{n}")

Routing accuracy: 17/17 = 100%

  directa            6/6
  informal           2/2
  ambigua            6/6
  fuera_de_alcance   2/2
  multi_intencion    1/1


In [3]:
# Agente evaluador (bonus): calidad de las respuestas, de 1 a 10
evaluadas = [r for r in resultados if r["evaluacion"]]

def promedio(campo):
    return sum(getattr(r["evaluacion"], campo) for r in evaluadas) / len(evaluadas)

print(f"Respuestas evaluadas: {len(evaluadas)} (las out_of_scope no se evalúan)\n")
for campo, nombre in [("score", "quality"), ("relevance", "relevance"),
                      ("completeness", "completeness"), ("accuracy", "accuracy")]:
    print(f"  {nombre:<13} promedio {promedio(campo):.1f} / 10")

print("\nLas 3 respuestas con menor calidad:")
for r in sorted(evaluadas, key=lambda r: r["evaluacion"].score)[:3]:
    e = r["evaluacion"]
    print(f"\n  #{r['id']} {r['query']}")
    print(f"  quality={e.score} completeness={e.completeness} | {e.reasoning}")

Respuestas evaluadas: 15 (las out_of_scope no se evalúan)

  quality       promedio 8.7 / 10
  relevance     promedio 9.5 / 10
  completeness  promedio 8.3 / 10
  accuracy      promedio 8.5 / 10

Las 3 respuestas con menor calidad:

  #16 Quiero pedir vacaciones y además mi laptop no prende
  quality=5 completeness=4 | La respuesta toca ambos temas, pero solo deriva de forma parcial y no indica un paso concreto para pedir vacaciones ni cómo contactar a TI o Mesa de Ayuda. Es coherente y prudente al no inventar procedimientos, aunque queda demasiado general para que el colaborador avance.

  #7 oye cuando me cae la grati de julio??
  quality=7 completeness=7 | Responde directamente cuándo se paga la gratificación de julio y añade una explicación útil sobre el cálculo. Sin embargo, no indica qué hacer si el colaborador necesita confirmar su caso específico ni ofrece un canal interno; además, la respuesta mezcla información general con una fuente genérica sin contexto.

  #4 La VPN se que

In [9]:
# Respuestas de las consultas ambiguas: las más difíciles del dataset
for r in resultados:
    if r["category"] == "ambigua":
        print(f"#{r['id']} {r['query']}  ->  {r['obtenido']}")
        print(r["respuesta"])
        print("-" * 80)

#9 Compré unos guantes de seguridad con mi plata porque no había en almacén, ¿me lo devuelven?  ->  finance
Sí, si compraste los guantes por una emergencia porque no había stock en almacén, puedes pedir el reembolso a Finanzas. Debes presentar el comprobante de compra y una constancia del supervisor de SSOMA de la obra que confirme que no había stock; además, el reembolso de EPP no requiere aprobación del gerente del área, solo del supervisor de SSOMA y de tu jefe directo.  
Fuente: Reembolso de EPP comprados por el colaborador.
--------------------------------------------------------------------------------
#10 La app de Andina360 no me deja marcar mi asistencia  ->  tech
Si la aplicación de Andina360 no te deja marcar tu asistencia, y el problema es técnico como que no abre, no reconoce la ubicación o muestra un error, debes reportarlo a la Mesa de Ayuda de TI. Si la app sí funciona pero la información de asistencia o saldo parece incorrecta, corresponde consultarlo con Recursos Huma

## 6. Integración con Langfuse

Cada consulta genera un **trace** en Langfuse con esta estructura:

```
support-request              (trace: consulta, respuesta final y scores)
├── orchestrator-routing     (span: intent, confidence, reasoning)
│   └── ChatOpenAI           (generation: prompt, respuesta, tokens, costo)
├── <intent>-agent           (agent: hr-agent / tech-agent / finance-agent)
│   ├── retrieval            (retriever: chunks recuperados con su score)
│   └── ChatOpenAI           (generation)
└── evaluator                (evaluator: califica la respuesta del 1 al 10)
    └── ChatOpenAI           (generation)
```

Las llamadas al LLM se registran automáticamente con el `CallbackHandler` de Langfuse para LangChain. Los spans se crean con `start_as_current_observation` (Langfuse v4). El evaluador registra cuatro **scores** en cada trace: `quality`, `relevance`, `completeness` y `accuracy`. Las consultas fuera de alcance no se evalúan.

**Cómo ver las trazas:** en cloud.langfuse.com → proyecto → **Tracing**, filtrar por *Is Root Observation = true* y por el tag `pruebas-v2` para ver solo las consultas de la sección 5.

In [10]:
from datetime import datetime, timedelta, timezone

desde = datetime.now(timezone.utc) - timedelta(hours=2)
observaciones = get_langfuse().api.observations.get_many(
    from_start_time=desde, name="support-request", limit=50
).data

print(f"Traces 'support-request' en las últimas 2 horas: {len(observaciones)}")
for o in sorted(observaciones, key=lambda o: o.start_time)[-5:]:
    print(f"  {o.start_time.strftime('%H:%M:%S')} | trace {o.trace_id[:8]} | latencia {o.latency:.2f}s")

Traces 'support-request' en las últimas 2 horas: 31
  03:44:59 | trace ff0dfd0d | latencia 2.95s
  03:45:02 | trace 2513a821 | latencia 2.83s
  03:45:04 | trace 7888aa8f | latencia 2.47s
  03:45:07 | trace 2f305fb4 | latencia 2.63s
  03:45:10 | trace be4d9cdb | latencia 2.25s
